# Experiment 2: Snapdragon Runtime Validation

## Objective

The objective of this experiment is to validate the Snapdragon runtime on the target Android device.

Unlike the previous experiment, this notebook does not build `llama.cpp`. Instead, it verifies that the generated binaries, shared libraries and Qualcomm runtime can execute successfully on the Samsung Galaxy S26 Ultra.

No model inference is performed in this experiment.

## Background

Experiment 1 successfully produced a Snapdragon-enabled build of `llama.cpp`.

Generated artifacts include:

- `llama-cli`
- `llama-server`
- `llama-bench`
- Qualcomm backend libraries

The next step is to deploy these artifacts to the Android device and verify that the runtime initializes correctly.

## Runtime Architecture

The runtime execution flow is shown below.

```text
llama-cli
      │
      ▼
libggml.so
      │
      ▼
libggml-hexagon.so
      │
      ▼
Qualcomm AI Runtime
      │
      ▼
Hexagon Driver
      │
      ▼
HTP Devices
```

This experiment validates each stage of the runtime before attempting model execution.

## Experiment Goal

The experiment is considered successful if the following objectives are achieved:

- Deployment package transferred successfully
- Executables launch correctly
- Shared libraries are loaded
- Qualcomm runtime initializes
- Hexagon backend is detected

Model inference is intentionally excluded from this experiment.

### Step 1: Verify Android Device

```bash
adb devices
```

### Step 2: Create Deployment Directory

```bash
adb shell "mkdir -p /data/local/tmp/llama"
```

### Step 3: Transfer Package

```bash
adb push llama.cpp/pkg-snapdragon/llama.cpp/. /data/local/tmp/llama/
```

### Step 4: Verify Deployment

```bash
adb shell "ls -R /data/local/tmp/llama"
```

## Configure Runtime

Configure the runtime library paths before executing the binaries.

```bash
adb shell
```

Then inside:
```bash
cd /data/local/tmp/llama

export LD_LIBRARY_PATH=$PWD/lib:/vendor/lib64:/system/lib64
export ADSP_LIBRARY_PATH=/vendor/lib/rfsa/adsp:/system/lib/rfsa/adsp:$PWD/lib
```

## Validate Runtime

```bash
./bin/llama-cli --help
```

Expected: 
```text
Usage: 
llama-cli ...
```

This tells us the executable starts and all required shared libraries are found.

## Backend Validation

```bash
./bin/llama-bench --help
```

Then:
```bash
./bin/llama-server --help
```
At this stage, we only verify that the binaries launch successfully.

## Runtime Observations

The deployment package was successfully transferred to the Samsung Galaxy S26 Ultra.

During runtime validation, the following observations were made sequentially.

### Observation 1

Initial execution produced:

```text
Permission denied
```

Cause:

- Executable permissions were not preserved after `adb push`.

Resolution:

```bash
chmod +x bin/*
```

---

### Observation 2

After correcting executable permissions:

```text
library "libllama-cli-impl.so" not found
```

Cause:

The runtime library search path was not configured.

Resolution:

```bash
export LD_LIBRARY_PATH=/data/local/tmp/llama/lib:/vendor/lib64:/system/lib64
```

---

### Observation 3

After configuring the runtime library path:

```text
cannot locate symbol
_ZNSt3__113__hash_memoryEPKvm
```

The executable now reaches Android's dynamic linker, indicating that the runtime initialization has progressed further.

The remaining issue appears to originate from runtime library resolution rather than deployment or executable permissions.

## Investigation Findings

The generated deployment package was inspected after installation.

The package correctly contains:

- libllama.so
- libllama-cli-impl.so
- libggml.so
- libggml-hexagon.so
- libggml-opencl.so
- libggml-htp-v73.so
- libggml-htp-v75.so
- libggml-htp-v79.so
- libggml-htp-v81.so

However, no C++ runtime library such as:

```text
libc++_shared.so
```

was included in the installation package.

Inspection of the official Snapdragon CMake preset also shows that no explicit Android STL configuration is specified.

This suggests that the remaining runtime issue is related to dynamic library resolution rather than compilation or deployment.

## Experiment Status

Current progress:

| Stage | Status |
|--------|--------|
| Snapdragon build | Completed |
| Android package generation | Completed |
| Deployment to device | Completed |
| Executable permissions | Resolved |
| Runtime library discovery | Resolved |
| Dynamic linker initialization | Reached |
| Hexagon runtime initialization | Pending |

At this stage the runtime reaches Android's dynamic linker before failing due to unresolved C++ runtime symbols.

Further investigation is required before backend validation and model execution.

## Research Notes

During this investigation, community reports were identified demonstrating successful execution of the Snapdragon backend on Android devices such as the OnePlus 12 (Snapdragon 8 Gen 3).

This indicates that Android execution is achievable and that the current issue is likely related to runtime configuration, packaging, or compatibility rather than lack of backend support.

Future work will focus on identifying the missing runtime dependency and completing Hexagon backend initialization.

## Current Status

The experiment successfully validated the deployment pipeline and identified the first runtime initialization issue.

The next phase will focus on resolving the dynamic linking problem before validating Qualcomm Hexagon backend execution.